# Suicide rates and regional population size: Spain and the United States, 2001–2021
### Reproducible figure pipeline for the paper

This notebook turns the raw CSV files into **every figure and every number quoted in the paper**. Nothing is edited by hand: *Run All* regenerates the whole set.

---

## 1. Research question and storyline

> **Are more populated regions associated with higher or lower suicide rates?**

The paper follows four steps, and each main figure answers one of them:

| Step | Question | Figure | File written |
|---|---|---|---|
| 1. Context | How did population and the national suicide rate evolve? | **Fig. 1** indexed national trends | `fig1_national_trends` |
| 2. Where | Where are rates high or low, region by region? | **Fig. 2** tile-grid maps | `fig2_tile_maps` |
| 3. Core result | Does the regional rate depend on population size? | **Fig. 3** scatter + population terciles | `fig3_rate_vs_population` |
| 4. Robustness | Are the differences larger than chance, and is the pattern stable over time? | **Fig. 4** funnel plots + annual correlation | `fig4_robustness` |
| Supplement | Who changed most? Full ranking with uncertainty | **Fig. S1** change map, **Fig. S2** ranked intervals | `figS1_change_map`, `figS2_ranked_rates` |

Only these figures are produced on purpose: each uses the chart type that makes its message easiest to read, and region names appear only where a reader needs them.

## 2. Repository layout and how to reproduce

```
your-repo/
├── README.md
├── requirements.txt
├── data/                  <- the four CSV files below (folder may also be called "datos")
│   ├── usa_population.csv
│   ├── usa_suicide_rate.csv
│   ├── spain_population.csv
│   └── spain_suicide_rate.csv
├── notebooks/
│   └── suicide_rates_paper_figures.ipynb      <- this notebook
├── figures/               <- created automatically (PDF + PNG of every figure)
└── results/               <- created automatically (CSV with the numbers quoted in the text)
```

1. Clone the repository.
2. `pip install -r requirements.txt`
3. Open this notebook and choose **Run All**.

**No path depends on your computer.** The notebook looks for the repository root by walking up from its own working directory until it finds the `data/` folder, and builds every path with `pathlib`. It works wherever the repository is cloned, on Windows, macOS or Linux.

## 3. Data

| File | One row per | Columns used | Source |
|---|---|---|---|
| `usa_suicide_rate.csv` | US state × year | `Year`, `Location`, `Deaths` | CDC (suicide deaths) |
| `usa_population.csv` | US state × year | `year`, `region_name`, `population` | U.S. Census Bureau |
| `spain_suicide_rate.csv` | autonomous community × year | `Periodo`, `Comunidades y Ciudades Autónomas`, `Total` | INE (suicide deaths) |
| `spain_population.csv` | autonomous community × year | `year`, `region_name`, `population` | see the data repository (cite the provider here) |

Coverage: 51 US regions (50 states + DC) and 19 Spanish regions (17 autonomous communities + Ceuta and Melilla), 2001–2021.

## 4. Methodological decisions (and why)

- **Always rates, never counts.** A larger region has more suicides only because it has more people, so counts against population would always rise together. Everything is *suicides per 100,000 inhabitants*, computed by us as `suicides / population × 100,000`. Rates are **crude** (no age adjustment) because the Spanish data have no age-adjusted series and the US file switches from crude to age-adjusted in 2014.
- **National series are indexed to 2001 = 100.** Population only grows over time, so a scatter of national rate against population would measure the shared time trend, not an effect of size.
- **The unit of the size analysis is the region, in a pooled period** (`PERIOD = (2017, 2021)`). Small regions (Ceuta, Melilla) have 0–6 suicides a year, so single-year rates are noise. Pooling: `rate = Σ suicides / Σ person-years × 100,000`.
- **Uncertainty.** Counts are rare events, so each regional rate carries an exact 95 % Poisson (Garwood) interval.
- **Association measures.** Spearman rank correlation ρ (robust to the very different region sizes) with a bootstrap 95 % interval; and a least-squares line of the rate on log₁₀(population) with its 95 % confidence band. Both are *descriptive and ecological*, not causal.
- **Population terciles** (small / medium / large, within each country) turn the continuous relation into a plain comparison of medians.
- **Funnel plot** compares each rate with the range expected from chance alone given its population (Wilson–Hilferty approximation of Poisson limits).
- **Reproducibility.** No randomness except the bootstrap and the jitter of the strip chart; both use fixed seeds.

## 5. How to read this notebook

Every code cell is preceded by a markdown cell that says **what it does and why**. Figures are followed by a **suggested caption** that can be pasted into the paper. Figure sizes are the **final printed sizes** (7.1 in wide), so text sizes in the file are the text sizes on the page: do not rescale figures in the manuscript.

## 6. Setup

### 6.1 Imports, library versions and switches

*What it does.* Imports the libraries (only `numpy`, `pandas`, `matplotlib`, `scipy`), prints their versions (useful to cite or to debug a different environment) and defines the switches that control the run.

- `SAVE_FIGURES`: write every figure as vector PDF (editable text) and 300-dpi PNG into `figures/`.
- `PERIOD`: years pooled for the regional analysis.
- `EARLY`, `LATE`: pooled windows used only in the supplementary change map.
- `N_BOOT`, `SEED`: bootstrap resamples and random seed (fixed, so results are identical on every run).

In [ ]:
import sys, textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import scipy
from matplotlib.colors import Normalize, LinearSegmentedColormap, to_rgb
from matplotlib.cm import ScalarMappable
from matplotlib.patches import FancyBboxPatch, Rectangle, Patch
from matplotlib.transforms import Bbox
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter, NullLocator, MaxNLocator
from scipy.stats import spearmanr, chi2, norm, t as tdist

# ----- switches -----
SAVE_FIGURES = True            # write PDF + PNG of every figure into <repo>/figures
PERIOD = (2017, 2021)          # years pooled for the regional analysis (first, last)
EARLY, LATE = (2001, 2003), (2019, 2021)   # pooled windows for the supplementary change map
N_BOOT, SEED = 5000, 2024      # bootstrap resamples and random seed

print(f"Python {sys.version.split()[0]} | numpy {np.__version__} | pandas {pd.__version__} | "
      f"matplotlib {mpl.__version__} | scipy {scipy.__version__}")

### 6.2 Paths (the reproducibility core)

*What it does.* Finds the repository root and defines every input/output path with `pathlib`, so no absolute path is ever written in the code.

1. Starting from the notebook's working directory, it walks **up** the folder tree until it finds a folder named `data` (or `datos`). That folder's parent is the repository root.
2. `DATA_DIR`, `FIG_DIR` and `RESULTS_DIR` are derived from the root. Output folders are created if missing.
3. It checks that the four CSV files exist and stops with a clear message if one is missing.

If you launch Jupyter from an unrelated folder, set `REPO_ROOT_OVERRIDE = Path("...")` once; otherwise leave it as `None`.

In [ ]:
REPO_ROOT_OVERRIDE = None                 # e.g. Path("/path/to/your-repo") if Jupyter is started elsewhere
DATA_FOLDER_NAMES = ("data", "datos")     # accepted names of the data folder

def find_repo_root(start=None):
    # Walk up from `start` (default: current working directory) until a data folder is found
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        for name in DATA_FOLDER_NAMES:
            if (folder / name).is_dir():
                return folder, folder / name
    raise FileNotFoundError(
        f"No '{'/'.join(DATA_FOLDER_NAMES)}' folder found in {start} or any parent folder. "
        "Open the notebook from inside the repository or set REPO_ROOT_OVERRIDE.")

if REPO_ROOT_OVERRIDE is not None:
    REPO_ROOT = Path(REPO_ROOT_OVERRIDE).resolve()
    DATA_DIR = next(REPO_ROOT / n for n in DATA_FOLDER_NAMES if (REPO_ROOT / n).is_dir())
else:
    REPO_ROOT, DATA_DIR = find_repo_root()

FIG_DIR = REPO_ROOT / "figures"
RESULTS_DIR = REPO_ROOT / "results"
FIG_DIR.mkdir(exist_ok=True); RESULTS_DIR.mkdir(exist_ok=True)

FILES = {
    "usa_pop": DATA_DIR / "usa_population.csv",
    "usa_sui": DATA_DIR / "usa_suicide_rate.csv",
    "esp_pop": DATA_DIR / "spain_population.csv",
    "esp_sui": DATA_DIR / "spain_suicide_rate.csv",
}
missing = [p.name for p in FILES.values() if not p.exists()]
assert not missing, f"Missing input files in {DATA_DIR}: {missing}"
print("Repository root:", REPO_ROOT.name, "| data folder:", DATA_DIR.name, "| all input files found")

## 7. Visual identity and plotting helpers

### 7.1 Palette, colour maps and typography

*What it does.* Sets the visual language used in **every** figure, so the paper looks like one piece:

- **United States = blue, Spain = purple**, orange = observations highlighted for a statistical reason, grey = uncertainty and reference elements.
- Within a country, three shades of its colour mark the **population tercile** (light = small, dark = large).
- One sequential colour map for rates (shared by both countries, so colours are comparable) and one diverging blue–orange map for differences.
- Typography sized for the printed page: 8 pt text, 7.5 pt ticks. Fonts are embedded as editable text in PDFs.

In [ ]:
FULL_W = 7.1    # width of a full-page-width figure, in inches (final printed size)

BLUE, PURPLE, ORANGE = "#356FA8", "#8064A2", "#D97745"
INK, MUTED, LIGHT = "#202A35", "#687582", "#DCE3E9"
PALE_GREY = "#F4F6F8"
COUNTRIES = ["United States", "Spain"]
COUNTRY_COLOR = {"United States": BLUE, "Spain": PURPLE}

CMAP_RATE = plt.get_cmap("YlGnBu")                                   # sequential: rates
CMAP_DIV = LinearSegmentedColormap.from_list(                         # diverging: blue below, orange above
    "blue_orange", [BLUE, "#7FA6CC", "#F7F7F5", "#EFA77F", ORANGE])

TIERS = ["Small", "Medium", "Large"]
TIER_F = {"Small": 0.42, "Medium": 0.70, "Large": 1.0}               # colour strength per tercile

mpl.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 8,
    "axes.titlesize": 9, "axes.labelsize": 8, "xtick.labelsize": 7.5, "ytick.labelsize": 7.5,
    "legend.fontsize": 7.5, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.edgecolor": LIGHT, "axes.linewidth": 0.7,
    "lines.linewidth": 1.6, "figure.facecolor": "white", "axes.facecolor": "white",
    "savefig.facecolor": "white", "figure.dpi": 130, "savefig.dpi": 300,
    "pdf.fonttype": 42, "ps.fonttype": 42,          # embed fonts as editable text
})

### 7.2 Generic helpers

*What it does.* Small functions reused by several figures:

- `tint` mixes a colour with white (used for the tercile shades).
- `clean_axes` applies the minimal editorial style (no top/right spines, light grid on one axis).
- `pop_formatter` writes populations as `100k`, `1M`, `10M`.
- `fmt_p` formats p-values; `panel_title` draws the bold panel letter plus the coloured panel title.
- `save_figure` writes PDF + PNG into `figures/` when `SAVE_FIGURES` is on.

In [ ]:
def tint(color, f):
    # Mix a colour with white: f = 1 keeps the colour, f = 0 gives white
    return tuple(1 - (1 - v) * f for v in to_rgb(color))

def clean_axes(ax, grid="y", left=True):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.spines["left"].set_color(LIGHT); ax.spines["bottom"].set_color(LIGHT)
    ax.spines["left"].set_visible(left)
    ax.tick_params(length=0, pad=3)
    ax.set_axisbelow(True); ax.grid(False)
    if grid in ("y", "both"): ax.grid(axis="y", color=LIGHT, lw=0.5, alpha=0.9)
    if grid in ("x", "both"): ax.grid(axis="x", color=LIGHT, lw=0.5, alpha=0.9)

def pop_formatter(v, _):
    if v >= 1_000_000: return f"{v/1_000_000:g}M"
    if v >= 1_000: return f"{v/1_000:g}k"
    return f"{v:g}"

def fmt_p(p):
    return "p < 0.001" if p < 0.001 else (f"p = {p:.3f}" if p < 0.01 else f"p = {p:.2f}")

def panel_title(ax, letter, text, color=INK, dx_pts=0):
    # Bold panel letter followed by a coloured title, just above the axes (offsets in points, so they do not depend on axes width)
    ax.annotate(letter, (0, 1.035), xycoords="axes fraction", xytext=(dx_pts, 0), textcoords="offset points",
                fontsize=9.5, fontweight="bold", va="bottom", ha="left", annotation_clip=False)
    ax.annotate(text, (0, 1.035), xycoords="axes fraction", xytext=(dx_pts + 13, 0), textcoords="offset points",
                fontsize=8.5, fontweight="bold", color=color, va="bottom", ha="left", annotation_clip=False)

def save_figure(fig, name):
    if SAVE_FIGURES:
        fig.savefig(FIG_DIR / f"{name}.pdf", bbox_inches="tight", pad_inches=0.03)
        fig.savefig(FIG_DIR / f"{name}.png", dpi=300, bbox_inches="tight", pad_inches=0.03)
        print(f"saved: {(FIG_DIR / name).relative_to(REPO_ROOT)}.pdf / .png")

### 7.3 Collision-free labels

*What it does.* `place_labels` names selected points in a scatter **without overlaps and without extra libraries**. For each label it tries positions around its point (increasing distance) and keeps the first one that (i) stays inside the axes, (ii) overlaps no other label and (iii) covers no data point. Far labels get a thin leader line.

It must be called **after** scales, limits and layout are final, because it measures text in screen coordinates.

In [ ]:
def place_labels(ax, x, y, labels, all_xy=None, fontsize=6.5, color=None):
    color = color or INK
    fig = ax.figure
    fig.canvas.draw()
    rend = fig.canvas.get_renderer()
    axbb = ax.get_window_extent(rend)
    pts = ax.transData.transform(np.column_stack(all_xy if all_xy is not None else (x, y)))
    halo = [pe.withStroke(linewidth=2.2, foreground="white")]
    dirs = [(1, 0, "left", "center"), (-1, 0, "right", "center"), (0, 1, "center", "bottom"), (0, -1, "center", "top"),
            (.7, .7, "left", "bottom"), (-.7, .7, "right", "bottom"), (.7, -.7, "left", "top"), (-.7, -.7, "right", "top")]
    cands = [(r, dx * r, dy * r, ha, va) for r in (4, 9, 15, 23, 32) for dx, dy, ha, va in dirs]
    placed = []
    for xi, yi, lab in zip(x, y, labels):
        chosen = None
        for r, dx, dy, ha, va in cands:
            tmp = ax.annotate(lab, (xi, yi), xytext=(dx, dy), textcoords="offset points", ha=ha, va=va, fontsize=fontsize)
            bb = tmp.get_window_extent(rend); tmp.remove()
            box = Bbox.from_extents(bb.x0 - 1.5, bb.y0 - 1, bb.x1 + 1.5, bb.y1 + 1)
            inside = axbb.x0 <= box.x0 and box.x1 <= axbb.x1 and axbb.y0 <= box.y0 and box.y1 <= axbb.y1
            hits = np.any((pts[:, 0] > box.x0 - 1.5) & (pts[:, 0] < box.x1 + 1.5) & (pts[:, 1] > box.y0 - 1.5) & (pts[:, 1] < box.y1 + 1.5))
            if inside and not hits and not any(box.overlaps(p) for p in placed):
                chosen = (r, dx, dy, ha, va, box); break
        if chosen is None:                                   # crowded: far up-right fallback
            r, dx, dy, ha, va = 32, 22, 22, "left", "bottom"
            tmp = ax.annotate(lab, (xi, yi), xytext=(dx, dy), textcoords="offset points", ha=ha, va=va, fontsize=fontsize)
            bb = tmp.get_window_extent(rend); tmp.remove()
            chosen = (r, dx, dy, ha, va, Bbox.from_extents(bb.x0 - 1.5, bb.y0 - 1, bb.x1 + 1.5, bb.y1 + 1))
        r, dx, dy, ha, va, box = chosen
        ax.annotate(lab, (xi, yi), xytext=(dx, dy), textcoords="offset points", ha=ha, va=va, fontsize=fontsize, color=color,
                    path_effects=halo, zorder=9, annotation_clip=False,
                    arrowprops=dict(arrowstyle="-", color=MUTED, lw=0.5, shrinkA=0, shrinkB=2) if r >= 15 else None)
        placed.append(box)

## 8. Data

### 8.1 Load, harmonise and compute rates

*What it does.*
1. Reads the four CSV files with the paths built above.
2. Gives US and Spanish data **the same four columns** (`year`, `region`, `suicides`, `population`).
3. Harmonises Spanish region names (the two Spanish files spell some communities differently; the suicide file also carries a numeric prefix like `01 Andalucía`).
4. Joins suicides and population with `validate="1:1"`, so pandas raises an error if any region-year is duplicated or ambiguous.
5. Computes `rate = suicides / population × 100,000`.
6. Runs **sanity checks** (expected number of regions and rows, no missing values, no negative counts). If the data were changed, these checks fail loudly instead of silently producing wrong figures.

In [ ]:
PopUSA = pd.read_csv(FILES["usa_pop"])      # population by state and year
SuiUSA = pd.read_csv(FILES["usa_sui"])      # suicide deaths by state and year
PopESP = pd.read_csv(FILES["esp_pop"])      # population by autonomous community and year
SuiESP = pd.read_csv(FILES["esp_sui"])      # suicide deaths by autonomous community and year

# Spanish names that differ between the two files
ESP_NAMES = {
    "03 Asturias, Principado de": "Principado de Asturias", "04 Balears, Illes": "Illes Balears",
    "08 Castilla - La Mancha": "Castilla-La Mancha", "13 Madrid, Comunidad de": "Comunidad de Madrid",
    "14 Murcia, Región de": "Región de Murcia", "15 Navarra, Comunidad Foral de": "Comunidad Foral de Navarra",
    "17 Rioja, La": "La Rioja", "18 Ceuta": "Ciudad de Ceuta", "19 Melilla": "Ciudad de Melilla",
}
esp_name = lambda s: ESP_NAMES.get(s, s[3:])         # all others only lose the "01 " prefix

usa = (SuiUSA.rename(columns={"Year": "year", "Location": "region", "Deaths": "suicides"})[["year", "region", "suicides"]]
       .merge(PopUSA.rename(columns={"region_name": "region"})[["year", "region", "population"]],
              on=["year", "region"], validate="1:1"))
esp = (SuiESP.rename(columns={"Periodo": "year", "Total": "suicides"})
       .assign(region=lambda t: t["Comunidades y Ciudades Autónomas"].map(esp_name))[["year", "region", "suicides"]]
       .merge(PopESP.rename(columns={"region_name": "region"})[["year", "region", "population"]],
              on=["year", "region"], validate="1:1"))

PANELS = {"United States": usa, "Spain": esp}
EXPECTED_REGIONS = {"United States": 51, "Spain": 19}
for name, p in PANELS.items():
    p["rate"] = p["suicides"] / p["population"] * 100_000            # suicides per 100,000 inhabitants
    assert p["region"].nunique() == EXPECTED_REGIONS[name], f"{name}: unexpected number of regions"
    assert len(p) == EXPECTED_REGIONS[name] * p["year"].nunique(), f"{name}: incomplete region-year grid"
    assert p[["suicides", "population"]].notna().all().all() and (p["suicides"] >= 0).all(), f"{name}: bad values"
    print(f"{name:14s} {p['region'].nunique()} regions x {p['year'].nunique()} years = {len(p)} rows "
          f"({p['year'].min()}-{p['year'].max()})")

### 8.2 Short labels and tile-grid geometry

*What it does.* Defines how regions are named inside figures and where each region sits in the **tile-grid map** (one square per region, placed roughly where it is on the map; US layout follows the common "statebins" arrangement). It needs no shapefile or external download, which keeps the repository self-contained.

Two safety checks verify that every region has exactly one tile and that no two tiles overlap.

In [ ]:
US_ABBR = {
    "Alabama":"AL","Alaska":"AK","Arizona":"AZ","Arkansas":"AR","California":"CA","Colorado":"CO","Connecticut":"CT",
    "Delaware":"DE","District of Columbia":"DC","Florida":"FL","Georgia":"GA","Hawaii":"HI","Idaho":"ID","Illinois":"IL",
    "Indiana":"IN","Iowa":"IA","Kansas":"KS","Kentucky":"KY","Louisiana":"LA","Maine":"ME","Maryland":"MD",
    "Massachusetts":"MA","Michigan":"MI","Minnesota":"MN","Mississippi":"MS","Missouri":"MO","Montana":"MT",
    "Nebraska":"NE","Nevada":"NV","New Hampshire":"NH","New Jersey":"NJ","New Mexico":"NM","New York":"NY",
    "North Carolina":"NC","North Dakota":"ND","Ohio":"OH","Oklahoma":"OK","Oregon":"OR","Pennsylvania":"PA",
    "Rhode Island":"RI","South Carolina":"SC","South Dakota":"SD","Tennessee":"TN","Texas":"TX","Utah":"UT",
    "Vermont":"VT","Virginia":"VA","Washington":"WA","West Virginia":"WV","Wisconsin":"WI","Wyoming":"WY",
}
ESP_SHORT = {
    "Andalucía":"Andalucía", "Aragón":"Aragón", "Principado de Asturias":"Asturias", "Illes Balears":"Baleares",
    "Canarias":"Canarias", "Cantabria":"Cantabria", "Castilla y León":"Castilla\ny León", "Castilla-La Mancha":"Castilla-\nLa Mancha",
    "Cataluña":"Cataluña", "Comunitat Valenciana":"C.\nValenciana", "Extremadura":"Extremadura", "Galicia":"Galicia",
    "Comunidad de Madrid":"Madrid", "Región de Murcia":"Murcia", "Comunidad Foral de Navarra":"Navarra",
    "País Vasco":"País\nVasco", "La Rioja":"La Rioja", "Ciudad de Ceuta":"Ceuta", "Ciudad de Melilla":"Melilla",
}
def short_label(country, region, oneline=True):
    s = US_ABBR[region] if country == "United States" else ESP_SHORT[region]
    return s.replace("-\n", "-").replace("\n", " ") if oneline else s

US_TILES_BY_ABBR = {                                         # abbreviation: (column, row)
    "AK":(0,0),"ME":(10,0),
    "WI":(5,1),"VT":(9,1),"NH":(10,1),
    "WA":(1,2),"ID":(2,2),"MT":(3,2),"ND":(4,2),"MN":(5,2),"IL":(6,2),"MI":(7,2),"NY":(8,2),"MA":(9,2),
    "OR":(1,3),"NV":(2,3),"WY":(3,3),"SD":(4,3),"IA":(5,3),"IN":(6,3),"OH":(7,3),"PA":(8,3),"NJ":(9,3),"CT":(10,3),
    "CA":(1,4),"UT":(2,4),"CO":(3,4),"NE":(4,4),"MO":(5,4),"KY":(6,4),"WV":(7,4),"VA":(8,4),"MD":(9,4),"RI":(10,4),
    "AZ":(2,5),"NM":(3,5),"KS":(4,5),"AR":(5,5),"TN":(6,5),"NC":(7,5),"SC":(8,5),"DC":(9,5),"DE":(10,5),
    "OK":(4,6),"LA":(5,6),"MS":(6,6),"AL":(7,6),"GA":(8,6),
    "HI":(0,7),"TX":(4,7),"FL":(9,7),
}
ABBR_TO_US = {v: k for k, v in US_ABBR.items()}
TILES = {
    "United States": {ABBR_TO_US[a]: pos for a, pos in US_TILES_BY_ABBR.items()},
    "Spain": {
        "Galicia":(0,0), "Principado de Asturias":(1,0), "Cantabria":(2,0), "País Vasco":(3,0), "Comunidad Foral de Navarra":(4,0),
        "Castilla y León":(1,1), "La Rioja":(3,1), "Aragón":(4,1), "Cataluña":(5,1),
        "Extremadura":(1,2), "Comunidad de Madrid":(2,2), "Castilla-La Mancha":(3,2), "Comunitat Valenciana":(4,2), "Illes Balears":(6,2),
        "Andalucía":(2,3), "Región de Murcia":(3,3),
        "Canarias":(0,4), "Ciudad de Ceuta":(1,4), "Ciudad de Melilla":(3,4),
    },
}
for c in COUNTRIES:
    assert set(TILES[c]) == set(PANELS[c]["region"].unique()), f"{c}: tile layout does not match the data"
    assert len(set(TILES[c].values())) == len(TILES[c]), f"{c}: two tiles share a cell"
print("Tile layouts OK:", {c: len(v) for c, v in TILES.items()})

## 9. Analysis (all numbers are computed here, before any plotting)

### 9.1 National series

*What it does.* Adds up all regions per year to get the national suicide count and population, computes the national rate and indexes both series to 2001 = 100. Used by **Fig. 1**.

In [ ]:
national = {}
for name, p in PANELS.items():
    n = p.groupby("year")[["suicides", "population"]].sum()
    n["rate"] = n["suicides"] / n["population"] * 100_000
    n["pop_index"] = n["population"] / n["population"].iloc[0] * 100      # 2001 = 100
    n["rate_index"] = n["rate"] / n["rate"].iloc[0] * 100
    national[name] = n

display(pd.concat({k: v[["population", "suicides", "rate", "pop_index", "rate_index"]].round(2) for k, v in national.items()}, axis=1).iloc[[0, 5, 10, 15, 20]])

### 9.2 Regional table for the pooled period, with Poisson intervals and terciles

*What it does.* For every region, over `PERIOD`:

- sums suicides and person-years, and computes the **pooled rate**;
- attaches an **exact 95 % Poisson interval** to the rate (`poisson_ci`);
- assigns the **population tercile** (Small / Medium / Large) within its country;
- computes the national pooled rate (`NATIONAL_RATE`) used as a reference line.

`period_table` is reused later for the early/late windows of the supplementary map. Used by **Figs. 2, 3, 4, S1, S2**.

In [ ]:
def poisson_ci(k, alpha=0.05):
    # Exact (Garwood) confidence interval for a Poisson count k
    k = np.asarray(k, dtype=float)
    lo = np.where(k > 0, chi2.ppf(alpha / 2, 2 * np.maximum(k, 1)) / 2, 0.0)
    hi = chi2.ppf(1 - alpha / 2, 2 * (k + 1)) / 2
    return lo, hi

def period_table(panel, y0, y1):
    # One row per region for the years y0..y1: pooled rate, mean population and exact 95 % interval
    w = panel[(panel["year"] >= y0) & (panel["year"] <= y1)]
    t = w.groupby("region").agg(suicides=("suicides", "sum"), person_years=("population", "sum"), pop=("population", "mean"))
    t["rate"] = t["suicides"] / t["person_years"] * 100_000
    lo, hi = poisson_ci(t["suicides"])
    t["rate_lo"], t["rate_hi"] = lo / t["person_years"] * 100_000, hi / t["person_years"] * 100_000
    return t

TABLES = {name: period_table(p, *PERIOD) for name, p in PANELS.items()}
for name, t in TABLES.items():
    t["tier"] = pd.qcut(t["pop"], 3, labels=TIERS).astype(str)                # population terciles within the country
    t["short"] = [short_label(name, r) for r in t.index]
NATIONAL_RATE = {n: t["suicides"].sum() / t["person_years"].sum() * 100_000 for n, t in TABLES.items()}
N_YEARS = PERIOD[1] - PERIOD[0] + 1
VMAX = float(np.nanpercentile(np.concatenate([t["rate"].values for t in TABLES.values()]), 97))   # shared upper colour limit
NORM_RATE = Normalize(vmin=0, vmax=VMAX)

for name, t in TABLES.items():
    print(f"{name}: {len(t)} regions, {int(t['suicides'].sum()):,} suicides in {PERIOD[0]}-{PERIOD[1]}; "
          f"national pooled rate {NATIONAL_RATE[name]:.2f} per 100,000; tercile sizes {t['tier'].value_counts()[TIERS].tolist()}")
display(TABLES["Spain"].sort_values("pop").round(2).head(4))

### 9.3 Association between population size and rate

*What it does.* For each country:

- **Spearman ρ** and its p-value between population and rate across regions, plus a **bootstrap 95 % interval** (regions are resampled `N_BOOT` times with a fixed seed);
- a **least-squares line** of the rate on log₁₀(population): slope = change in rate per tenfold larger population, with its 95 % interval, and the confidence band of the fitted mean (`ols_line`);
- the **median rate per tercile**.

Both measures are reported on purpose: the rank correlation is less sensitive to extreme regions than the line. Used by **Fig. 3**.

In [ ]:
def spearman_boot_ci(x, y, n_boot=N_BOOT, seed=SEED):
    # Percentile bootstrap interval of Spearman's rho (resampling regions)
    rng = np.random.default_rng(seed)
    x, y = np.asarray(x), np.asarray(y)
    idx = rng.integers(0, len(x), size=(n_boot, len(x)))
    r = np.array([spearmanr(x[i], y[i])[0] for i in idx])
    return tuple(np.nanpercentile(r, [2.5, 97.5]))

def ols(t):
    # Rate regressed on log10(population); 95 % interval of the slope; Spearman with bootstrap interval
    x, y = np.log10(t["pop"].values), t["rate"].values
    n = len(x)
    b, a = np.polyfit(x, y, 1)
    res = y - (a + b * x)
    s2, sxx = (res ** 2).sum() / (n - 2), ((x - x.mean()) ** 2).sum()
    se_b, tc = np.sqrt(s2 / sxx), tdist.ppf(0.975, n - 2)
    rho, p = spearmanr(t["pop"], t["rate"])
    return dict(a=a, b=b, ci=(b - tc * se_b, b + tc * se_b), s2=s2, sxx=sxx, xm=x.mean(), n=n, tc=tc,
                resid=res, rho=rho, p=p, rho_ci=spearman_boot_ci(t["pop"], t["rate"]))

def ols_line(m, xs_log):
    # Fitted line and 95 % confidence band of the mean
    yhat = m["a"] + m["b"] * xs_log
    se = np.sqrt(m["s2"] * (1 / m["n"] + (xs_log - m["xm"]) ** 2 / m["sxx"]))
    return yhat, yhat - m["tc"] * se, yhat + m["tc"] * se

FIT = {n: ols(t) for n, t in TABLES.items()}
TIER_MEDIAN = {n: t.groupby("tier")["rate"].median() for n, t in TABLES.items()}
for n, m in FIT.items():
    print(f"{n:14s} Spearman rho = {m['rho']:+.2f} [{m['rho_ci'][0]:+.2f}, {m['rho_ci'][1]:+.2f}], {fmt_p(m['p'])} | "
          f"slope per x10 population = {m['b']:+.1f} [{m['ci'][0]:+.1f}, {m['ci'][1]:+.1f}] | "
          f"median rate small/large tercile = {TIER_MEDIAN[n]['Small']:.1f}/{TIER_MEDIAN[n]['Large']:.1f}")

### 9.4 Robustness: funnel limits and year-by-year correlation

*What it does.*

- **Funnel.** If every region had the national rate, observed rates would scatter only because suicides are rare events, and the scatter would shrink as population grows. `funnel_limits` gives the 95 % and 99.8 % limits expected from chance for a region of a given population; `funnel_outside` flags the regions beyond the 99.8 % limits.
- **Year by year.** The Spearman correlation across regions is recomputed for every single year, with an approximate 95 % interval (Fisher transform), to see whether the pattern is stable over 2001–2021.

Used by **Fig. 4**.

In [ ]:
def funnel_limits(pop, p0, n_years, z):
    # Wilson-Hilferty approximation to the Poisson limits for a rate per 100,000 (p0 = national rate)
    lam = np.maximum(p0 * np.asarray(pop) * n_years / 100_000, 1e-9)         # expected number of suicides
    return p0 * np.maximum(1 - 1 / (9 * lam) + z / (3 * np.sqrt(lam)), 0) ** 3

def funnel_outside(t, p0, n_years, z=3.09):
    up, dn = funnel_limits(t["pop"], p0, n_years, z), funnel_limits(t["pop"], p0, n_years, -z)
    out = (t["rate"] > up) | (t["rate"] < dn)
    dist = np.where(t["rate"] > p0, (t["rate"] - p0) / (up - p0), (p0 - t["rate"]) / (p0 - dn))   # distance to the limit
    return out, pd.Series(dist, index=t.index)

FUNNEL = {n: funnel_outside(t, NATIONAL_RATE[n], N_YEARS) for n, t in TABLES.items()}

rows = []
for name, p in PANELS.items():
    for year, g in p.groupby("year"):
        n = len(g)
        rho, pv = spearmanr(g["population"], g["rate"])
        rc = np.clip(rho, -0.999, 0.999)
        se = np.sqrt((1 + rc ** 2 / 2) / (n - 3))                              # standard error of atanh(rho)
        lo, hi = np.tanh(np.arctanh(rc) + np.array([-1.96, 1.96]) * se)
        rows.append(dict(country=name, year=year, rho=rho, p=pv, lo=lo, hi=hi))
RHO_YEAR = pd.DataFrame(rows)

for n in COUNTRIES:
    r = RHO_YEAR[RHO_YEAR["country"] == n]
    print(f"{n:14s} regions outside the 99.8 % funnel: {int(FUNNEL[n][0].sum())}/{len(TABLES[n])} | "
          f"annual rho from {r['rho'].min():+.2f} to {r['rho'].max():+.2f}; interval excludes 0 in {int(((r['lo'] > 0) | (r['hi'] < 0)).sum())}/{len(r)} years")

## 10. Figures

### Figure 1 — National context: population and suicide rate, indexed to 2001

*What it shows.* For each country, total population (black) and national crude suicide rate (coloured), both relative to 2001. The **shaded gap** is how far the rate moved away from population growth. This is the context for the regional analysis: in the US the rate rose much more than the population; in Spain it fluctuated around its 2001 level.

*Design choices.* Same vertical scale in both panels so the countries can be compared; direct labels at the line ends instead of a legend; no title inside the figure (the caption goes in the manuscript).

*What the cell does.* Reads `national` (section 9.1), draws two panels with `sharey=True`, places each end label above or below its line depending on which series ends higher, and saves the figure.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(FULL_W, 2.65), sharey=True)
lo = min(min(n["pop_index"].min(), n["rate_index"].min()) for n in national.values()) - 4
hi = max(max(n["pop_index"].max(), n["rate_index"].max()) for n in national.values()) + 9
for ax, name, letter in zip(axes, COUNTRIES, "ab"):
    n, c = national[name], COUNTRY_COLOR[name]
    ax.fill_between(n.index, n["pop_index"], n["rate_index"], color=c, alpha=0.11, lw=0)
    ax.axhline(100, color=MUTED, lw=0.7, ls=(0, (3, 3)), zorder=0)
    ax.plot(n.index, n["pop_index"], color=INK, lw=1.6)
    ax.plot(n.index, n["rate_index"], color=c, lw=2.0)
    rate_on_top = n["rate_index"].iloc[-1] >= n["pop_index"].iloc[-1]
    for col, label, color, above in [("pop_index", "Population", INK, not rate_on_top), ("rate_index", "Suicide rate", c, True)]:
        v = n[col].iloc[-1]
        ax.annotate(f"{label} ×{v/100:.2f}", (n.index[-1], v), xytext=(-1, 5 if above else -5), textcoords="offset points",
                    ha="right", va="bottom" if above else "top", fontsize=7, fontweight="bold", color=color)
    ax.set_xlim(2001, 2021); ax.set_xticks(range(2001, 2022, 5)); ax.set_ylim(lo, hi)
    ax.set_xlabel("Year")
    panel_title(ax, letter, name, c)
    clean_axes(ax, "y")
axes[0].set_ylabel("Index (2001 = 100)")
fig.subplots_adjust(left=0.075, right=0.985, top=0.87, bottom=0.18, wspace=0.13)
plt.show()
save_figure(fig, "fig1_national_trends")

**Suggested caption — Figure 1.** *National population (black) and crude suicide rate per 100,000 inhabitants (coloured), each relative to its 2001 value, for (a) the United States and (b) Spain, 2001–2021. The shaded area is the difference between the two series. Sources: CDC and U.S. Census Bureau (United States); INE (Spain).*

### Figure 2 — Where are suicide rates high or low? Tile-grid maps

*What it shows.* The pooled crude rate of every region (`PERIOD`), one tile per region, placed roughly where the region is. The name and the value are written **inside each tile**, so no region needs a separate label or legend lookup.

*Design choices.*
- On a normal map, Texas is huge while Rhode Island, DC or Ceuta are invisible. In a tile grid every region has the same visual weight.
- One **sequential colour scale shared by both countries**, so equal colours mean equal rates in (a) and (b). Values above the 97th percentile of regional rates saturate (the colour bar has an arrow).
- Tile size is slightly larger for Spain only so that the full Spanish names fit; tile size carries no meaning.

*What the cell does.* Defines `tile_map` (draws one grid) and `tile_figure` (lays out the two grids in inches, adds the colour bar). `tile_figure` is reused for the supplementary change map. Then it builds Figure 2.

In [ ]:
def lum(rgba):
    r, g, b = rgba[:3]
    return 0.299 * r + 0.587 * g + 0.114 * b

def tile_map(ax, country, values, cmap, norm, value_fmt):
    pos = TILES[country]
    ncols, nrows = max(c for c, r in pos.values()) + 1, max(r for c, r in pos.values()) + 1
    us = country == "United States"
    for region, (cx, cy) in pos.items():
        v = values[region]
        col = cmap(norm(v))
        ax.add_patch(FancyBboxPatch((cx + 0.04, cy + 0.04), 0.92, 0.92, boxstyle="round,pad=0,rounding_size=0.09", fc=col, ec="none"))
        tc = "white" if lum(col) < 0.52 else INK
        lab = short_label(country, region, oneline=False)
        two = "\n" in lab
        ax.text(cx + 0.5, cy + (0.40 if us else (0.34 if two else 0.40)), lab, ha="center", va="center",
                fontsize=7 if us else 5.4, fontweight="bold", color=tc, linespacing=0.95)
        ax.text(cx + 0.5, cy + (0.76 if us else 0.80), value_fmt(v), ha="center", va="center", fontsize=5.6 if us else 5.8, color=tc)
    ax.set_xlim(0, ncols); ax.set_ylim(nrows, 0); ax.axis("off")

def tile_figure(values, cmap, norm, value_fmt, cbar_label, extend, ticks=None, ticklabels=None):
    size = {"United States": 0.285, "Spain": 0.54}               # tile side in inches
    gap, top_pad, cb_h = 0.18, 0.30, 0.66
    dims = {c: (max(x for x, y in TILES[c].values()) + 1, max(y for x, y in TILES[c].values()) + 1) for c in COUNTRIES}
    h = {c: dims[c][1] * size[c] for c in COUNTRIES}
    H = top_pad + max(h.values()) + cb_h
    fig = plt.figure(figsize=(FULL_W, H))
    x0 = 0.0
    for letter, c in zip("ab", COUNTRIES):
        w = dims[c][0] * size[c]
        ax = fig.add_axes([x0 / FULL_W, (H - top_pad - h[c]) / H, w / FULL_W, h[c] / H])
        tile_map(ax, c, values[c], cmap, norm, value_fmt)
        panel_title(ax, letter, c, COUNTRY_COLOR[c])
        x0 += w + gap
    cax = fig.add_axes([(FULL_W - 2.6) / 2 / FULL_W, 0.30 / H, 2.6 / FULL_W, 0.09 / H])
    cb = fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), cax=cax, orientation="horizontal", extend=extend)
    cb.outline.set_visible(False); cb.ax.tick_params(length=0, labelsize=6.5, pad=2)
    cb.set_label(cbar_label, fontsize=7, labelpad=3)
    if ticks is not None:
        cb.set_ticks(ticks)
        if ticklabels is not None: cb.set_ticklabels(ticklabels)
    return fig

fig = tile_figure({c: TABLES[c]["rate"] for c in COUNTRIES}, CMAP_RATE, NORM_RATE, lambda v: f"{v:.1f}",
                  f"Suicides per 100,000 inhabitants, {PERIOD[0]}–{PERIOD[1]}", "max")
plt.show()
save_figure(fig, "fig2_tile_maps")

**Suggested caption — Figure 2.** *Crude suicide rate per 100,000 inhabitants pooled over 2017–2021 for (a) the 50 US states and the District of Columbia and (b) the 17 Spanish autonomous communities plus Ceuta and Melilla. Each region is one tile, placed approximately where it lies on the map (Alaska, Hawaii, the Canary Islands, Ceuta and Melilla in free cells); the number inside is the rate. The colour scale is shared by both countries and saturates above its upper limit. Tile size has no meaning.*

### Figure 3 — Main result: suicide rate and regional population size

*What it shows.* Each point is a region. **Left of each pair (scatter):** pooled rate against population (log scale) with the fitted trend, its 95 % confidence band, each region's exact 95 % Poisson interval (thin vertical bars) and the Spearman correlation. **Right (strip chart):** the same regions grouped into **population terciles**, with the median (black bar), the interquartile range (shaded box) and every region (dot).

*Design choices.*
- The scatter gives the continuous relation *with its uncertainty*; the strip chart gives the same message in plain language ("small regions vs. large regions"), which is easier to quote in the text.
- Dot shade is the tercile, consistently with the rest of the paper.
- Only the regions farthest from the trend and the most populated ones are named; labels are placed automatically without overlaps (`place_labels`).

*What the cell does.* Draws the scatter and strip chart for each country (`association_pair`), chooses which regions to name (`pick_labels`), places the labels after the layout is fixed and saves the figure.

In [ ]:
def minus(v, d=2, sign=True):
    return (f"{v:+.{d}f}" if sign else f"{v:.{d}f}").replace("-", "−")

def pick_labels(t, m, n_resid, n_big):
    # Regions farthest from the fit plus the most populated ones (no duplicates; order = priority)
    far = pd.Series(m["resid"], index=t.index).abs().nlargest(n_resid).index
    return list(dict.fromkeys(list(far) + list(t["pop"].nlargest(n_big).index)))

def association_pair(ax, ax_strip, country, letter):
    t, m, c = TABLES[country], FIT[country], COUNTRY_COLOR[country]
    x, y = t["pop"], t["rate"]
    ax.set_xscale("log")
    edges = [x.min() * 0.72]                                         # tercile boundaries as vertical bands
    for a, b in [("Small", "Medium"), ("Medium", "Large")]:
        edges.append(np.sqrt(t.loc[t.tier == a, "pop"].max() * t.loc[t.tier == b, "pop"].min()))
    edges.append(x.max() * 1.45)
    ax.axvspan(edges[1], edges[2], color=PALE_GREY, lw=0, zorder=0)
    for i, tier in enumerate(TIERS):
        ax.text(np.sqrt(edges[i] * edges[i + 1]), 0.02, tier, transform=ax.get_xaxis_transform(), ha="center", va="bottom", fontsize=6.3, color=MUTED, style="italic")
    ax.set_xlim(edges[0], edges[-1])
    xs = np.logspace(np.log10(x.min()), np.log10(x.max()), 120)
    yhat, lo, hi = ols_line(m, np.log10(xs))
    ax.fill_between(xs, lo, hi, color=c, alpha=0.14, lw=0, zorder=1)
    ax.plot(xs, yhat, color=c, lw=1.6, zorder=3)
    ax.errorbar(x, y, yerr=[y - t["rate_lo"], t["rate_hi"] - y], fmt="none", ecolor=tint(c, 0.38), elinewidth=0.7, zorder=2)
    ax.scatter(x, y, s=26, c=[tint(c, TIER_F[v]) for v in t["tier"]], edgecolors=c, linewidths=0.7, zorder=4)
    ax.set_ylim(0, t["rate_hi"].max() * 1.36)                       # headroom for the statistics box
    ax.yaxis.set_major_locator(MaxNLocator(nbins=6, integer=True))
    ax.xaxis.set_major_formatter(FuncFormatter(pop_formatter)); ax.xaxis.set_minor_locator(NullLocator())
    ax.set_xlabel("Population (log scale)"); ax.set_ylabel("Suicides per 100,000 inhabitants")
    panel_title(ax, letter, country, c)
    ax.text(0.985, 0.985, f"Spearman ρ = {minus(m['rho'])}  [{minus(m['rho_ci'][0])}, {minus(m['rho_ci'][1])}]\n{fmt_p(m['p'])}\n"
                         f"Fit: {minus(m['b'], 1)} per ×10 population\n[95% CI {minus(m['ci'][0], 1)}, {minus(m['ci'][1], 1)}]",
            transform=ax.transAxes, ha="right", va="top", fontsize=6.2, linespacing=1.4,
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=LIGHT, lw=0.6), zorder=8)
    clean_axes(ax, "y")

    rng = np.random.default_rng(3)                                   # strip chart (shares the y axis)
    for i, tier in enumerate(TIERS):
        g = t[t["tier"] == tier]
        q1, med, q3 = np.percentile(g["rate"], [25, 50, 75])
        ax_strip.add_patch(Rectangle((i - 0.34, q1), 0.68, q3 - q1, fc=tint(c, 0.13), ec="none", zorder=1))
        ax_strip.scatter(i + rng.uniform(-0.2, 0.2, len(g)), g["rate"], s=17, color=tint(c, TIER_F[tier]), edgecolors=c, lw=0.6, zorder=3)
        ax_strip.hlines(med, i - 0.34, i + 0.34, color=INK, lw=1.8, zorder=4)
        ax_strip.text(i, med, f"{med:.1f}", ha="center", va="bottom", fontsize=6.3, fontweight="bold", zorder=5,
                      path_effects=[pe.withStroke(linewidth=2, foreground="white")])
    ax_strip.set_xlim(-0.5, 2.5); ax_strip.set_ylim(ax.get_ylim())
    ax_strip.set_xticks(range(3)); ax_strip.set_xticklabels([f"{tr}\nn={int((t['tier']==tr).sum())}" for tr in TIERS], fontsize=5.8)
    ax_strip.tick_params(labelleft=False)
    ax_strip.text(0.0, 1.035, "By tercile", transform=ax_strip.transAxes, fontsize=7, color=MUTED, va="bottom")
    clean_axes(ax_strip, "y", left=False)

fig = plt.figure(figsize=(FULL_W, 3.3))
outer = fig.add_gridspec(1, 2, left=0.065, right=0.995, top=0.875, bottom=0.17, wspace=0.2)
scatter_axes = {}
for k, country in enumerate(COUNTRIES):
    inner = outer[0, k].subgridspec(1, 2, width_ratios=[3.3, 1.8], wspace=0.05)
    ax, ax_s = fig.add_subplot(inner[0]), fig.add_subplot(inner[1])
    association_pair(ax, ax_s, country, "ab"[k])
    scatter_axes[country] = ax
for country, (n_res, n_big) in {"United States": (6, 3), "Spain": (5, 2)}.items():     # name only the notable regions
    t = TABLES[country]
    names = pick_labels(t, FIT[country], n_res, n_big)
    place_labels(scatter_axes[country], t.loc[names, "pop"], t.loc[names, "rate"], t.loc[names, "short"],
                 all_xy=(t["pop"].values, t["rate"].values), fontsize=6.3)
plt.show()
save_figure(fig, "fig3_rate_vs_population")

**Suggested caption — Figure 3.** *Crude suicide rate and regional population size, pooled over 2017–2021, for (a) US states and (b) Spanish autonomous communities. Left of each pair: each dot is a region (shade = population tercile within the country); thin bars are exact 95 % Poisson intervals; the line is the least-squares fit of the rate on log₁₀ population with its 95 % confidence band. Spearman's ρ is given with a bootstrap 95 % interval. Right of each pair: the same regions by tercile, with the median (bar), interquartile range (box) and individual regions (dots). Only regions farthest from the fit and the most populated ones are named. Associations are ecological and descriptive.*

### Figure 4 — Robustness: is it more than chance, and is it stable over time?

*What it shows.*
- **(a, b) Funnel plots.** Rate against population, with the range expected from chance alone around the national rate (dark band = 95 %, light band = 99.8 %). Orange points are outside the 99.8 % limits. Small regions are allowed to deviate more because they have few suicides.
- **(c) Correlation year by year.** Spearman ρ between population and rate across regions, for each year 2001–2021, with an approximate 95 % interval. A band that contains 0 means that, that year, the data do not show a monotonic relation.

*How to read it.* (a, b) tell us that most regional differences are **larger than chance** (so they are real differences, not just small-number noise); (c) tells us whether the size–rate association **depends on the period chosen**.

*What the cell does.* Draws the two funnels from `TABLES`/`FUNNEL` (section 9.4), names the most extreme regions with `place_labels`, draws the annual correlation from `RHO_YEAR`, adds a shared key and saves the figure.

In [ ]:
fig = plt.figure(figsize=(FULL_W, 5.35))
gs_top = fig.add_gridspec(1, 2, left=0.075, right=0.99, top=0.935, bottom=0.575, wspace=0.2)     # funnels
gs_bot = fig.add_gridspec(1, 1, left=0.075, right=0.89, top=0.375, bottom=0.085)               # annual correlation
ax_f = [fig.add_subplot(gs_top[0, 0]), fig.add_subplot(gs_top[0, 1])]
ax_r = fig.add_subplot(gs_bot[0, 0])

# (a, b) funnel plots
for ax, country, letter in zip(ax_f, COUNTRIES, "ab"):
    t, p0, c = TABLES[country], NATIONAL_RATE[country], COUNTRY_COLOR[country]
    out, dist = FUNNEL[country]
    xs = np.logspace(np.log10(t["pop"].min() * 0.75), np.log10(t["pop"].max() * 1.3), 250)
    ax.set_xscale("log")
    ax.fill_between(xs, funnel_limits(xs, p0, N_YEARS, -3.09), funnel_limits(xs, p0, N_YEARS, 3.09), color=tint(c, 0.10), lw=0, zorder=0)
    ax.fill_between(xs, funnel_limits(xs, p0, N_YEARS, -1.96), funnel_limits(xs, p0, N_YEARS, 1.96), color=tint(c, 0.25), lw=0, zorder=1)
    ax.axhline(p0, color=INK, lw=0.9, zorder=2)
    ax.scatter(t.loc[~out, "pop"], t.loc[~out, "rate"], s=22, color=c, edgecolors="white", linewidths=0.5, zorder=4)
    ax.scatter(t.loc[out, "pop"], t.loc[out, "rate"], s=24, color=ORANGE, edgecolors="white", linewidths=0.5, zorder=5)
    ax.set_xlim(xs[0], xs[-1]); ax.set_ylim(0, t["rate"].max() * 1.1)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=6, integer=True))
    ax.xaxis.set_major_formatter(FuncFormatter(pop_formatter)); ax.xaxis.set_minor_locator(NullLocator())
    ax.set_xlabel("Population (log scale)"); ax.set_ylabel("Suicides per 100,000 inhabitants")
    ax.text(0.985, 0.03, f"{int(out.sum())} of {len(t)} regions outside 99.8 % limits", transform=ax.transAxes, ha="right", fontsize=6.5, fontweight="bold", color=ORANGE)
    ax.text(0.015, p0, f"national {p0:.1f}", transform=ax.get_yaxis_transform(), va="bottom", fontsize=6.2, color=INK)
    panel_title(ax, letter, country, c)
    clean_axes(ax, "y")

# (c) annual correlation
ax_r.axhline(0, color=INK, lw=0.8, ls=(0, (4, 3)))
for country in COUNTRIES:
    r, c = RHO_YEAR[RHO_YEAR["country"] == country], COUNTRY_COLOR[country]
    ax_r.fill_between(r["year"], r["lo"], r["hi"], color=c, alpha=0.13, lw=0)
    ax_r.plot(r["year"], r["rho"], color=c, lw=1.8, marker="o", ms=3, markeredgecolor="white", markeredgewidth=0.4)
    ax_r.annotate(country, (r["year"].iloc[-1], r["rho"].iloc[-1]), xytext=(5, 0), textcoords="offset points", color=c, fontsize=7, fontweight="bold", va="center", annotation_clip=False)
ax_r.set_xlim(2001, 2021); ax_r.set_ylim(-0.85, 0.85); ax_r.set_xticks(range(2001, 2022, 2))
ax_r.set_ylabel("Spearman ρ\n(population vs. rate)"); ax_r.set_xlabel("Year")
ax_r.text(0.005, 0.97, "larger regions have higher rates", transform=ax_r.transAxes, fontsize=6.5, color=MUTED, va="top")
ax_r.text(0.005, 0.03, "larger regions have lower rates", transform=ax_r.transAxes, fontsize=6.5, color=MUTED, va="bottom")
panel_title(ax_r, "c", "Year-by-year association", INK)
clean_axes(ax_r, "y")

key = [Patch(fc=tint(INK, 0.25), ec="none", label="95 % limits expected by chance"), Patch(fc=tint(INK, 0.10), ec="none", label="99.8 % limits"),
       Line2D([], [], marker="o", ls="", color=ORANGE, ms=4, label="region outside 99.8 % limits"), Line2D([], [], color=INK, lw=0.9, label="national rate")]
fig.legend(handles=key, loc="center", bbox_to_anchor=(0.53, 0.465), ncol=4, frameon=False, fontsize=6.5, handlelength=1.4, columnspacing=1.2)
for ax, country in zip(ax_f, COUNTRIES):
    t = TABLES[country]; far = FUNNEL[country][1][FUNNEL[country][0]].nlargest(6).index
    place_labels(ax, t.loc[far, "pop"], t.loc[far, "rate"], t.loc[far, "short"], all_xy=(t["pop"].values, t["rate"].values), fontsize=6.2)
plt.show()
save_figure(fig, "fig4_robustness")

**Suggested caption — Figure 4.** *Robustness of the regional pattern. (a, b) Funnel plots for US states and Spanish autonomous communities (2017–2021): the bands are the 95 % (dark) and 99.8 % (light) limits expected from Poisson variation around the national rate; orange points fall outside the 99.8 % limits, and the most extreme ones are named. (c) Spearman correlation between regional population and suicide rate in each year, with approximate 95 % intervals.*

## 11. Supplementary figures

### Figure S1 — Which regions changed most? Tile-grid map of the change

*What it shows.* The same tile grid as Figure 2, but the colour and the number now give the **change in the pooled rate** between the `EARLY` and `LATE` windows (three years each, to limit noise from small counts). Orange = the rate rose; blue = it fell.

*Why a supplement.* It shows that the US increase is nationwide (almost every state is orange) while Spain is mixed; it does not bear on the size question, so it supports the narrative without taking a main figure.

*What the cell does.* Computes `LATE − EARLY` pooled rates per region and reuses `tile_figure` with the diverging colour map (scale capped at ±8).

In [ ]:
CHANGE = {c: period_table(PANELS[c], *LATE)["rate"] - period_table(PANELS[c], *EARLY)["rate"] for c in COUNTRIES}
fig = tile_figure(CHANGE, CMAP_DIV, Normalize(-8, 8), lambda v: "0.0" if round(v, 1) == 0 else f"{v:+.1f}",
                  f"Change in suicides per 100,000 inhabitants, {EARLY[0]}–{EARLY[1]} to {LATE[0]}–{LATE[1]}", "both",
                  ticks=[-8, -4, 0, 4, 8], ticklabels=["≤ −8", "−4", "0", "+4", "≥ +8"])
plt.show()
save_figure(fig, "figS1_change_map")

**Suggested caption — Figure S1.** *Change in the pooled crude suicide rate per 100,000 inhabitants between 2001–2003 and 2019–2021 for (a) US states and (b) Spanish regions. Orange: increase; blue: decrease. The colour scale is capped at ±8.*

### Figure S2 — Full ranking with uncertainty (caterpillar plot)

*What it shows.* Every region ordered by pooled rate, with its **exact 95 % Poisson interval**. Names are on the axis, so nothing needs to be labelled over the data. A whisker that crosses the dashed national rate means the region's rate cannot be told apart from the national one given its number of suicides. Dot shade is the population tercile and the right-hand column gives the population.

*Why a supplement.* It is the "table as a figure": a reader who wants the value of a specific region finds it here, while the main text keeps only the message.

*What the cell does.* Draws the US panel at full height and the Spanish panel with the same row height (using a `height_ratios` grid), then adds a short "how to read" key.

In [ ]:
def caterpillar(ax, country, letter):
    t = TABLES[country].sort_values("rate", ascending=False)
    c, n, y = COUNTRY_COLOR[country], len(t), np.arange(len(t))
    xmax = t["rate_hi"].max() * 1.06
    ax.hlines(y, 0, xmax, color=LIGHT, lw=0.4, zorder=0)
    ax.hlines(y, t["rate_lo"], t["rate_hi"], color=tint(c, 0.5), lw=1.5, zorder=2)
    ax.scatter(t["rate"], y, s=18, c=[tint(c, TIER_F[v]) for v in t["tier"]], edgecolors=c, linewidths=0.6, zorder=4)
    nat = NATIONAL_RATE[country]
    ax.axvline(nat, color=INK, lw=0.9, ls=(0, (4, 3)), zorder=1)
    ax.text(nat, 1.005, f"national {nat:.1f}", transform=ax.get_xaxis_transform(), ha="center", va="bottom", fontsize=6.3, fontweight="bold")
    ax.set_yticks(y)
    ax.set_yticklabels([r if country == "United States" else short_label(country, r) for r in t.index], fontsize=6.3 if country == "United States" else 6.8, color=INK)
    ax.set_ylim(n - 0.5, -0.5); ax.set_xlim(0, xmax)
    ax.set_xlabel("Suicides per 100,000 inhabitants")
    for yi, pop in zip(y, t["pop"]):
        ax.text(1.012, yi, f"{pop/1e6:.1f}M" if pop >= 1e6 else f"{pop/1e3:.0f}k", transform=ax.get_yaxis_transform(), va="center", ha="left", fontsize=5.8, color=MUTED)
    ax.text(1.012, 1.005, "Pop.", transform=ax.transAxes, ha="left", va="bottom", fontsize=6.3, color=MUTED)
    panel_title(ax, letter, country, c, dx_pts=-76)
    clean_axes(ax, "x", left=False)

fig = plt.figure(figsize=(FULL_W, 8.9))
gs = fig.add_gridspec(2, 2, width_ratios=[1.15, 1], height_ratios=[19, 32], left=0.15, right=0.965, top=0.955, bottom=0.05, wspace=0.80, hspace=0.0)
ax_us, ax_es, ax_key = fig.add_subplot(gs[:, 0]), fig.add_subplot(gs[0, 1]), fig.add_subplot(gs[1, 1])
caterpillar(ax_us, "United States", "a"); caterpillar(ax_es, "Spain", "b")
ax_key.axis("off")
ax_key.text(0.0, 0.92, "How to read", fontsize=7.5, fontweight="bold", va="top", transform=ax_key.transAxes)
ax_key.text(0.0, 0.86, "Dot: pooled crude rate.\nWhisker: exact 95 % Poisson interval.\nDashed line: national pooled rate.\n"
                        "A whisker crossing the line is\ncompatible with the national rate.\nSmall regions have long whiskers\nbecause they have few suicides.",
            fontsize=6.5, va="top", linespacing=1.5, transform=ax_key.transAxes)
ax_key.text(0.0, 0.52, "Dot shade: population tercile\nwithin the country", fontsize=6.5, va="bottom", linespacing=1.4, transform=ax_key.transAxes)
for i, tr in enumerate(TIERS):
    ax_key.scatter([0.06 + 0.24 * i], [0.46], s=26, color=tint(INK, TIER_F[tr] * 0.8), edgecolors=INK, linewidths=0.6, transform=ax_key.transAxes)
    ax_key.text(0.06 + 0.24 * i, 0.425, tr, ha="center", va="top", fontsize=6.3, transform=ax_key.transAxes)
plt.show()
save_figure(fig, "figS2_ranked_rates")

**Suggested caption — Figure S2.** *Pooled crude suicide rate per 100,000 inhabitants in 2017–2021 for every (a) US state and (b) Spanish region, ranked from highest to lowest. Bars are exact 95 % Poisson intervals; the dashed line is the national rate; dot shade is the population tercile within the country; the right-hand column gives the mean population.*

## 12. Numbers quoted in the paper

*What it does.* Collects, in one table, every number the text of the paper needs (national growth, pooled rates, correlations with their intervals, fitted slopes, tercile medians, funnel counts, annual-correlation summary, extreme regions). The table is shown here and written to `results/key_numbers.csv`, so the text and the figures always come from the same computation and can be re-checked by anyone who clones the repository.

In [ ]:
rows = []
def add(country, quantity, value):
    rows.append({"country": country, "quantity": quantity, "value": value})

for n in COUNTRIES:
    nat, t, m = national[n], TABLES[n], FIT[n]
    ry = RHO_YEAR[RHO_YEAR["country"] == n]
    add(n, "National crude rate 2001 (per 100,000)", round(nat["rate"].iloc[0], 2))
    add(n, "National crude rate 2021 (per 100,000)", round(nat["rate"].iloc[-1], 2))
    add(n, "Rate growth 2001-2021 (x)", round(nat["rate"].iloc[-1] / nat["rate"].iloc[0], 2))
    add(n, "Population growth 2001-2021 (x)", round(nat["population"].iloc[-1] / nat["population"].iloc[0], 2))
    add(n, f"Pooled national rate {PERIOD[0]}-{PERIOD[1]} (per 100,000)", round(NATIONAL_RATE[n], 2))
    add(n, "Regions", len(t))
    add(n, "Spearman rho (population vs rate)", round(m["rho"], 2))
    add(n, "Spearman rho, bootstrap 95% CI low", round(m["rho_ci"][0], 2))
    add(n, "Spearman rho, bootstrap 95% CI high", round(m["rho_ci"][1], 2))
    add(n, "Spearman p-value", float(f"{m['p']:.3g}"))
    add(n, "Fit: change in rate per x10 population", round(m["b"], 2))
    add(n, "Fit slope, 95% CI low", round(m["ci"][0], 2))
    add(n, "Fit slope, 95% CI high", round(m["ci"][1], 2))
    for tier in TIERS:
        add(n, f"Median regional rate, {tier.lower()} tercile", round(TIER_MEDIAN[n][tier], 2))
    add(n, "Regions outside 99.8% funnel limits", int(FUNNEL[n][0].sum()))
    add(n, "Annual rho: minimum", round(ry["rho"].min(), 2))
    add(n, "Annual rho: maximum", round(ry["rho"].max(), 2))
    add(n, "Years with annual-rho interval excluding 0", f"{int(((ry['lo'] > 0) | (ry['hi'] < 0)).sum())} of {len(ry)}")
    add(n, "Highest pooled regional rate", f"{t['rate'].idxmax()} ({t['rate'].max():.1f})")
    add(n, "Lowest pooled regional rate", f"{t['rate'].idxmin()} ({t['rate'].min():.1f})")
    add(n, "Largest rate increase (early to late)", f"{CHANGE[n].idxmax()} ({CHANGE[n].max():+.1f})")

KEY_NUMBERS = pd.DataFrame(rows)
KEY_NUMBERS.to_csv(RESULTS_DIR / "key_numbers.csv", index=False)
display(KEY_NUMBERS.pivot(index="quantity", columns="country", values="value").loc[KEY_NUMBERS["quantity"].drop_duplicates()])
print("saved:", (RESULTS_DIR / "key_numbers.csv").relative_to(REPO_ROOT))

## 13. Interpretation, limitations and reproducibility checklist

**Reading the results together.** Figure 1 gives the context; Figure 2 shows where rates are high or low; Figure 3 is the core comparison between region size and rate; Figure 4 shows whether the pattern is more than noise and whether it is stable over time. Report the Spearman correlation and the tercile medians as the main numbers, and the fitted slope as a complement.

**Limitations to state in the paper.**
- Rates are **crude** (not age-adjusted), so differences in age structure between regions and countries are not removed.
- Regional population size is not the same as **rurality or population density**; other regional factors (income, age, health services, firearms access in the US) are not modelled.
- The associations are **ecological** (about regions, not individuals) and **descriptive**, not causal.
- Spain has only 19 regional units, so its estimates are imprecise; Ceuta and Melilla rest on very few suicides. The linear fit can be pulled by them, which is why the rank correlation is reported alongside it.
- Tile-grid maps trade geographic accuracy for legibility.

**Reproducibility checklist.**
- [x] All paths built with `pathlib` from the repository root found at run time; none is machine-specific.
- [x] Only `numpy`, `pandas`, `matplotlib`, `scipy`; versions printed at the top.
- [x] Fixed random seed for the bootstrap and the strip-chart jitter.
- [x] Sanity checks on the data fail loudly if the files change.
- [x] Every number in the text is written to `results/key_numbers.csv`; every figure to `figures/` as PDF (vector, editable text) and 300-dpi PNG.
- [x] Figures are produced at their final printed size (7.1 in wide).